# Survivor Count Sweep Diagnostics

Run HAPPO through `scripts/diagnose_joint_happo.py` and heuristic approaches through `scripts/diagnose_joint_baseline_strategies.py` for an active-survivor-count sweep, or provide an existing `FILELIST` of diagnostic JSON files to skip rerunning diagnostics. The notebook compares the core joint performance metrics:

- scout recall: mean and std
- confirmation recall: mean and std
- full-confirm success: rate and binary std
- final confidence: mean and std
- final coverage: mean and std
- scout/confirm/coverage/confidence AUC

Edit the sweep settings in the first code cell. Each survivor count selects its own checkpoint from `CHECKPOINTS_BY_SURVIVOR_COUNT`, while `STRATEGIES` selects the HAPPO and heuristic approaches to compare. If `FILELIST` is empty, the notebook runs only missing diagnostics, with up to `MAX_PARALLEL_RUNS` jobs running concurrently. If `FILELIST` is non-empty, it reads those files directly and does not launch diagnostics.


In [ ]:
# Survivor-count sweep configuration. Each count can use a differently trained checkpoint.
MODLABEL = 'uav4_ugv3_area_1sqkm_malibu_grid256_steps900_fire_survivors'
NSTEPS_PER_EPISODE = 900
SEED_START = 1000
SEED_END = 1099

# Sweep over the number of active true survivors in each episode.
# The checkpoint's survivor-slot count is preserved by default. Set SURVIVOR_SLOT_COUNT
# only when running without a checkpoint or when you intentionally want to override it.
SURVIVOR_COUNTS = [5, 10, 15, 20]
SURVIVOR_SLOT_COUNT = None
STRATEGIES = ['happo','ant_colony','lawnmower']

# Use None for a count to select the newest HAPPO checkpoint under f'happo_{MODLABEL}_{count}'.
CHECKPOINTS_BY_SURVIVOR_COUNT = {
    5: 'results/harl_runs/wildfire/wildfire_search/happo/happo_uav4_ugv3_area_1sqkm_malibu_grid256_steps900_fire_survivors_5/seed-00001-2026-07-18-23-56-28/models',
    10: 'results/harl_runs/wildfire/wildfire_search/happo/happo_uav4_ugv3_area_1sqkm_malibu_grid256_steps900_fire_survivors_10/seed-00001-2026-07-19-00-15-35/models',
    15: 'results/harl_runs/wildfire/wildfire_search/happo/happo_uav4_ugv3_area_1sqkm_malibu_grid256_steps900_fire_survivors_20/seed-00001-2026-07-19-00-16-34/models',
    20: 'results/harl_runs/wildfire/wildfire_search/happo/happo_uav4_ugv3_area_1sqkm_malibu_grid256_steps900_fire_survivors_20/seed-00001-2026-07-19-00-16-34/models',
}

RUN_DIAGNOSTICS = True
FORCE_RERUN_DIAGNOSTICS = False
# Use 1 for sequential execution; increase carefully because each diagnostic is CPU intensive.
MAX_PARALLEL_RUNS = 1
EXTRA_DIAGNOSTIC_ARGS = []

# Optional: provide existing diagnostic JSON outputs here to skip rerunning diagnostics.
FILELIST = []
#FILELIST = [
#     'outputs/survivors_load/happo_uav4_ugv3_area_1sqkm_malibu_grid256_steps900_fire_survivors_5_seeds_1000_1099.json',
#     'outputs/survivors_load/happo_uav4_ugv3_area_1sqkm_malibu_grid256_steps900_fire_survivors_10_seeds_1000_1099.json',
#     'outputs/survivors_load/happo_uav4_ugv3_area_1sqkm_malibu_grid256_steps900_fire_survivors_15_seeds_1000_1099.json',
#     'outputs/survivors_load/happo_uav4_ugv3_area_1sqkm_malibu_grid256_steps900_fire_survivors_20_seeds_1000_1099.json',
#]

# Optional display labels for FILELIST entries. Leave empty to derive approach/count labels.
LABELS = []


In [ ]:
import json
import re
import shlex
import subprocess
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

HAPPO_SCRIPT = Path('scripts/diagnose_joint_happo.py')
BASELINE_SCRIPT = Path('scripts/diagnose_joint_baseline_strategies.py')
CWD = Path.cwd().resolve()
if all((CWD / script).is_file() for script in (HAPPO_SCRIPT, BASELINE_SCRIPT)):
    PROJECT_ROOT = CWD
elif all((CWD.parent / script).is_file() for script in (HAPPO_SCRIPT, BASELINE_SCRIPT)):
    PROJECT_ROOT = CWD.parent
else:
    PROJECT_ROOT = Path('..').resolve()

OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'survivors_load'
RESULTS_ROOT = PROJECT_ROOT / 'results/harl_runs/wildfire/wildfire_search/happo'
STRATEGY_LABELS = {
    'happo': 'HAPPO',
    'ant_colony': 'ACO',
    'lawnmower': 'Lawnmower',
    'random_walk': 'Random Walk',
    'random_action': 'Random Action',
    'highest_confidence': 'Highest Confidence',
}


def _strategy_tag(strategy):
    return str(strategy).strip().lower().replace('-', '_')


def _diagnostic_prefix(strategy, checkpoint):
    strategy_tag = _strategy_tag(strategy)
    if strategy_tag == 'happo':
        return [
            sys.executable,
            str(PROJECT_ROOT / HAPPO_SCRIPT),
            '--checkpoint-dir', str(checkpoint),
        ]
    return [
        sys.executable,
        str(PROJECT_ROOT / BASELINE_SCRIPT),
        '--strategy', strategy_tag,
        '--happo-checkpoint', str(checkpoint),
    ]


def _resolve_checkpoint_for_count(count):
    count = int(count)
    if count not in CHECKPOINTS_BY_SURVIVOR_COUNT:
        raise KeyError(f'No checkpoint configured for survivor count {count}')
    configured = CHECKPOINTS_BY_SURVIVOR_COUNT[count]
    if configured is not None:
        path = Path(configured).expanduser()
        if not path.is_absolute():
            path = PROJECT_ROOT / path
        if not path.is_dir():
            raise FileNotFoundError(f'Checkpoint directory not found for n={count}: {path}')
        return path.resolve()

    run_label = f'happo_{MODLABEL}_{count}'
    candidates = sorted((RESULTS_ROOT / run_label).glob('seed-*/models'), key=lambda p: p.stat().st_mtime)
    if not candidates:
        raise FileNotFoundError(
            f'No checkpoint found under {RESULTS_ROOT / run_label}. '
            f'Set CHECKPOINTS_BY_SURVIVOR_COUNT[{count}] explicitly.'
        )
    return candidates[-1].resolve()


def _survivor_count_value(count):
    return int(count)


def _diagnostic_output_paths(strategy, count):
    stem = f'{_strategy_tag(strategy)}_{MODLABEL}_{int(count)}_dropout0p0_seeds_{SEED_START}_{SEED_END}'
    return OUTPUT_DIR / f'{stem}.json', OUTPUT_DIR / f'{stem}.png'


def _resolve_json_file(path):
    path = Path(path).expanduser()
    candidates = [path]
    if not path.is_absolute():
        candidates.extend([PROJECT_ROOT / path, PROJECT_ROOT / 'notebooks' / path])
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()
    return path.resolve() if path.is_absolute() else (PROJECT_ROOT / path).resolve()



In [ ]:
def _run_diagnostic_job(strategy, count, cmd, log_path):
    with log_path.open('w') as log:
        subprocess.run(
            cmd,
            cwd=PROJECT_ROOT,
            check=True,
            stdout=log,
            stderr=subprocess.STDOUT,
        )
    return strategy, count


survivor_counts = [_survivor_count_value(count) for count in SURVIVOR_COUNTS]
strategies = [_strategy_tag(strategy) for strategy in STRATEGIES]
if len(survivor_counts) != len(set(survivor_counts)):
    raise ValueError('SURVIVOR_COUNTS contains duplicates')
if any(count <= 0 for count in survivor_counts):
    raise ValueError('SURVIVOR_COUNTS must contain only positive counts')
if len(strategies) != len(set(strategies)):
    raise ValueError('STRATEGIES contains duplicates')
if not strategies:
    raise ValueError('STRATEGIES must contain at least one approach')
if int(MAX_PARALLEL_RUNS) < 1:
    raise ValueError('MAX_PARALLEL_RUNS must be at least 1')

RUN_IDENTITIES = []
if FILELIST:
    JSON_FILES = [_resolve_json_file(path) for path in FILELIST]
    print('Using provided file list; diagnostics will not be run.')
else:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    seeds = [str(seed) for seed in range(SEED_START, SEED_END + 1)]
    JSON_FILES = []
    jobs = []

    total_runs = len(strategies) * len(survivor_counts)
    run_index = 0
    for strategy in strategies:
        for count in survivor_counts:
            run_index += 1
            checkpoint = _resolve_checkpoint_for_count(count)
            json_output, plots_output = _diagnostic_output_paths(strategy, count)
            JSON_FILES.append(json_output)
            RUN_IDENTITIES.append((strategy, count))
            should_run = RUN_DIAGNOSTICS and (
                FORCE_RERUN_DIAGNOSTICS or not json_output.is_file()
            )
            run_label = f'{strategy}/n={count}'
            if not should_run:
                print(f'[{run_index}/{total_runs}] Using existing {run_label}:', json_output)
                continue

            cmd = _diagnostic_prefix(strategy, checkpoint)
            cmd.extend([
                '--steps', str(NSTEPS_PER_EPISODE),
                '--seeds', *seeds,
                '--active-survivors-min', str(count),
                '--active-survivors-max', str(count),
                '--json-output', str(json_output),
                '--plots-output', str(plots_output),
            ])
            if SURVIVOR_SLOT_COUNT is not None:
                cmd.extend(['--n-survivors', str(int(SURVIVOR_SLOT_COUNT))])
            cmd.extend(str(arg) for arg in EXTRA_DIAGNOSTIC_ARGS)
            log_path = json_output.with_suffix('.log')
            jobs.append((strategy, count, cmd, log_path))
            print(f'[{run_index}/{total_runs}] Queued {run_label}:', shlex.join(cmd))
            print('Checkpoint:', checkpoint)
            print('Log:', log_path)

    if jobs:
        workers = min(int(MAX_PARALLEL_RUNS), len(jobs))
        print(f'Running {len(jobs)} missing diagnostic(s) with {workers} worker(s).')
        with ThreadPoolExecutor(max_workers=workers) as executor:
            futures = {
                executor.submit(
                    _run_diagnostic_job, strategy, count, cmd, log_path
                ): (strategy, count, log_path)
                for strategy, count, cmd, log_path in jobs
            }
            for future in as_completed(futures):
                strategy, count, log_path = futures[future]
                future.result()
                print(f'Finished {strategy}/n={count}; log={log_path}')

missing = [path for path in JSON_FILES if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing diagnostic JSON files:\n' + '\n'.join(str(p) for p in missing))

print(f'Loaded file list: {len(JSON_FILES)} JSON file(s)')
for path in JSON_FILES:
    print(' -', path.relative_to(PROJECT_ROOT) if path.is_relative_to(PROJECT_ROOT) else path)


In [ ]:
def _summary_value(summary, *keys, default=float('nan')):
    for key in keys:
        if key in summary:
            return summary[key]
    return default


def _payload_scenario(payload):
    scenario = payload.get('scenario')
    if isinstance(scenario, dict) and scenario:
        return scenario
    scenario_kwargs = payload.get('scenario_kwargs')
    if isinstance(scenario_kwargs, dict) and scenario_kwargs:
        return scenario_kwargs
    metadata = payload.get('metadata', {})
    scenario_kwargs = metadata.get('scenario_kwargs') if isinstance(metadata, dict) else None
    return scenario_kwargs if isinstance(scenario_kwargs, dict) else {}


def _success_rate(summary):
    rate = _summary_value(summary, 'full_confirm_success_rate', 'success_rate')
    if pd.isna(rate):
        percent = _summary_value(summary, 'full_confirm_success_percent')
        rate = percent / 100.0 if not pd.isna(percent) else float('nan')
    return rate


def _success_std(summary, rate):
    if pd.isna(rate):
        return float('nan')
    episodes = _summary_value(summary, 'episodes')
    if pd.isna(episodes) or episodes <= 0:
        return float('nan')
    return float(np.sqrt(max(rate * (1.0 - rate), 0.0)))


def _scenario_number(scenario, key):
    value = scenario.get(key, float('nan'))
    return float(value) if isinstance(value, (int, float, np.floating)) and np.isfinite(value) else float('nan')


def _run_identity(path, payload, file_index):
    if len(RUN_IDENTITIES) == len(JSON_FILES):
        strategy, count = RUN_IDENTITIES[file_index]
        return _strategy_tag(strategy), int(count)

    metadata = payload.get('metadata', {})
    payload_strategy = payload.get('strategy') or payload.get('approach')
    if payload_strategy is None and isinstance(metadata, dict):
        payload_strategy = metadata.get('strategy') or metadata.get('approach')
    stem = path.stem.lower()
    configured_strategies = [_strategy_tag(strategy) for strategy in STRATEGIES]
    strategy = _strategy_tag(payload_strategy) if payload_strategy else next(
        (name for name in configured_strategies if stem.startswith(f'{name}_')),
        'unknown',
    )

    scenario = _payload_scenario(payload)
    active_min = _scenario_number(scenario, 'active_survivors_min')
    active_max = _scenario_number(scenario, 'active_survivors_max')
    if np.isfinite(active_min) and np.isfinite(active_max) and active_min == active_max:
        count = int(active_min)
    else:
        match = re.search(r'_survivors_(\d+)_seeds_', stem)
        count = int(match.group(1)) if match else float('nan')
    return strategy, count


def _label_for_identity(strategy, count, path):
    strategy_label = STRATEGY_LABELS.get(strategy, strategy.replace('_', ' ').title())
    if isinstance(count, (int, float, np.floating)) and np.isfinite(count):
        return f'{strategy_label} | n={int(count)}'
    return f'{strategy_label} | {path.stem}'


def _finite(values):
    return [float(v) for v in values if isinstance(v, (int, float, np.floating)) and np.isfinite(v)]


def _mean_std(values):
    values = _finite(values)
    if not values:
        return float('nan'), float('nan')
    return float(np.mean(values)), float(np.std(values))


def _episode_steps(row, scenario):
    for key in ('episode_steps', 'max_steps'):
        value = row.get(key)
        if isinstance(value, (int, float)) and value > 0:
            return int(value)
    value = scenario.get('max_steps')
    if isinstance(value, (int, float)) and value > 0:
        return int(value)
    return int(NSTEPS_PER_EPISODE)


def _active_survivor_indices(row):
    explicit = row.get('active_survivor_indices')
    if isinstance(explicit, list):
        return [int(idx) for idx in explicit]
    mask = row.get('active_survivor_mask')
    if isinstance(mask, list):
        return [idx for idx, active in enumerate(mask) if bool(active)]
    steps = row.get('first_scout_steps') or row.get('first_confirm_steps') or []
    count = row.get('active_survivors', row.get('survivors', len(steps)))
    try:
        count = int(count)
    except (TypeError, ValueError):
        count = len(steps)
    return list(range(min(count, len(steps))))


def _auc_from_first_steps(row, scenario, key):
    first_steps = row.get(key)
    if not isinstance(first_steps, list):
        return float('nan')
    steps = max(_episode_steps(row, scenario), 1)
    indices = _active_survivor_indices(row)
    if not indices:
        return 1.0
    total = 0.0
    for idx in indices:
        if idx >= len(first_steps):
            continue
        first_step = first_steps[idx]
        if first_step is None:
            continue
        try:
            first_step = float(first_step)
        except (TypeError, ValueError):
            continue
        if first_step <= 0:
            total += 1.0
        elif first_step <= steps:
            total += (steps - first_step + 1.0) / steps
    return float(total / max(len(indices), 1))


def _row_values(rows, *keys):
    values = []
    for row in rows:
        for key in keys:
            value = row.get(key)
            if isinstance(value, (int, float, np.floating)) and np.isfinite(value):
                values.append(float(value))
                break
    return values


def _summary_or_rows(summary, rows, mean_key, std_key, *row_keys, fallback=None):
    mean = _summary_value(summary, mean_key)
    std = _summary_value(summary, std_key)
    if not pd.isna(mean):
        return float(mean), float(std) if not pd.isna(std) else float('nan')
    values = _row_values(rows, *row_keys)
    if not values and fallback is not None:
        values = [fallback(row) for row in rows]
    return _mean_std(values)


if LABELS and len(LABELS) != len(JSON_FILES):
    raise ValueError(f'LABELS has {len(LABELS)} entries, but JSON_FILES has {len(JSON_FILES)} files')

records = []
for file_index, path in enumerate(JSON_FILES):
    payload = json.loads(path.read_text())
    summary = payload.get('summary', {})
    rows = payload.get('rows', [])
    scenario = _payload_scenario(payload)
    success_rate = _success_rate(summary)
    scout_auc_mean, scout_auc_std = _summary_or_rows(
        summary,
        rows,
        'mean_scout_auc',
        'std_scout_auc',
        'scout_auc',
        fallback=lambda row, scenario=scenario: _auc_from_first_steps(row, scenario, 'first_scout_steps'),
    )
    confirm_auc_mean, confirm_auc_std = _summary_or_rows(
        summary,
        rows,
        'mean_confirm_auc',
        'std_confirm_auc',
        'confirm_auc',
        'confirmation_auc',
        fallback=lambda row, scenario=scenario: _auc_from_first_steps(row, scenario, 'first_confirm_steps'),
    )
    coverage_auc_mean, coverage_auc_std = _summary_or_rows(
        summary,
        rows,
        'mean_coverage_auc',
        'std_coverage_auc',
        'coverage_auc',
    )
    confidence_auc_mean, confidence_auc_std = _summary_or_rows(
        summary,
        rows,
        'mean_confidence_auc',
        'std_confidence_auc',
        'confidence_auc',
    )
    active_mean, active_std = _mean_std(_row_values(rows, 'active_survivors', 'survivors'))
    metadata_active_min = _scenario_number(scenario, 'active_survivors_min')
    metadata_active_max = _scenario_number(scenario, 'active_survivors_max')
    metadata_slots = _scenario_number(scenario, 'n_survivors')
    strategy, configured_survivors = _run_identity(path, payload, file_index)
    if isinstance(configured_survivors, (int, float, np.floating)) and np.isfinite(configured_survivors):
        survivor_x = configured_survivors
    elif np.isfinite(metadata_active_min) and np.isfinite(metadata_active_max) and metadata_active_min == metadata_active_max:
        survivor_x = metadata_active_min
    elif np.isfinite(active_mean):
        survivor_x = active_mean
    else:
        survivor_x = metadata_slots
    records.append({
        'file': str(path),
        'label': str(LABELS[file_index]) if LABELS else _label_for_identity(strategy, configured_survivors, path),
        'strategy': strategy,
        'episodes': _summary_value(summary, 'episodes', default=len(rows) if rows else float('nan')),
        'survivor_slots': metadata_slots,
        'active_survivors_min': metadata_active_min,
        'active_survivors_max': metadata_active_max,
        'active_survivors_mean': active_mean,
        'active_survivors_std': active_std,
        'configured_survivor_count': configured_survivors,
        'survivor_x': survivor_x,
        'scout_recall_mean': _summary_value(summary, 'mean_scout_recall'),
        'scout_recall_std': _summary_value(summary, 'std_scout_recall'),
        'confirm_recall_mean': _summary_value(summary, 'mean_confirm_recall'),
        'confirm_recall_std': _summary_value(summary, 'std_confirm_recall'),
        'success_mean': success_rate,
        'success_std': _success_std(summary, success_rate),
        'success_count': _summary_value(summary, 'full_confirm_success_count'),
        'final_confidence_mean': _summary_value(summary, 'mean_final_confidence'),
        'final_confidence_std': _summary_value(summary, 'std_final_confidence'),
        'final_coverage_mean': _summary_value(summary, 'mean_final_coverage_fraction'),
        'final_coverage_std': _summary_value(summary, 'std_final_coverage_fraction'),
        'scout_auc_mean': scout_auc_mean,
        'scout_auc_std': scout_auc_std,
        'confirm_auc_mean': confirm_auc_mean,
        'confirm_auc_std': confirm_auc_std,
        'coverage_auc_mean': coverage_auc_mean,
        'coverage_auc_std': coverage_auc_std,
        'confidence_auc_mean': confidence_auc_mean,
        'confidence_auc_std': confidence_auc_std,
    })

metrics = pd.DataFrame.from_records(records)
metrics


## Compact Table

The table below formats each metric as `mean +/- std` for quick comparison. 
AUC values are time-integrated performance scores; newer JSON files read them directly, 
and older files backfill scout/confirm AUC from first scout/confirm steps when available.


In [ ]:
def _pm(mean, std):
    if pd.isna(mean):
        return 'n/a'
    if pd.isna(std):
        return f'{mean:.3f}'
    return f'{mean:.3f} +/- {std:.3f}'


compact = pd.DataFrame({
    'label': metrics['label'],
    'episodes': metrics['episodes'].astype('Int64'),
    'survivors': [
        _pm(mean, std)
        for mean, std in zip(metrics['active_survivors_mean'], metrics['active_survivors_std'])
    ],
    'scout recall': [
        _pm(mean, std)
        for mean, std in zip(metrics['scout_recall_mean'], metrics['scout_recall_std'])
    ],
    'confirm recall': [
        _pm(mean, std)
        for mean, std in zip(metrics['confirm_recall_mean'], metrics['confirm_recall_std'])
    ],
    'success': [
        _pm(mean, std)
        for mean, std in zip(metrics['success_mean'], metrics['success_std'])
    ],
    'success count': metrics['success_count'].astype('Int64'),
    'final confidence': [
        _pm(mean, std)
        for mean, std in zip(metrics['final_confidence_mean'], metrics['final_confidence_std'])
    ],
    'final coverage': [
        _pm(mean, std)
        for mean, std in zip(metrics['final_coverage_mean'], metrics['final_coverage_std'])
    ],
    'scout AUC': [
        _pm(mean, std)
        for mean, std in zip(metrics['scout_auc_mean'], metrics['scout_auc_std'])
    ],
    'confirm AUC': [
        _pm(mean, std)
        for mean, std in zip(metrics['confirm_auc_mean'], metrics['confirm_auc_std'])
    ],
    'coverage AUC': [
        _pm(mean, std)
        for mean, std in zip(metrics['coverage_auc_mean'], metrics['coverage_auc_std'])
    ],
    'confidence AUC': [
        _pm(mean, std)
        for mean, std in zip(metrics['confidence_auc_mean'], metrics['confidence_auc_std'])
    ],
}).set_index('label')

compact


## Plots

For survivor-count sweeps, the notebook plots `Confirm AUC` over the number of survivors with one line per approach. If no survivor-count axis is available, it falls back to grouped AUC bars for file-list comparisons.


In [ ]:
PAPER_BG = '#f2ebd3ff'
GRID_COLOR = '#cfc7ad'
TEXT_GREEN = '#23562f'
TEXT_DARK = '#2f2a24'
CONFIRM_COLOR = '#8f2418'
SCOUT_COLOR = '#c94a27'
AUC_METRIC_SPECS = [
    ('confirm_auc', 'Confirm AUC', CONFIRM_COLOR, 's'),
    ('scout_auc', 'Scout AUC', SCOUT_COLOR, 'o'),
    ('confidence_auc', 'Confidence AUC', '#1f5a35', '^'),
    ('coverage_auc', 'Coverage AUC', '#5f812f', 'v'),
]


plot_df = metrics.copy()
if 'survivor_x' not in plot_df.columns:
    plot_df['survivor_x'] = plot_df.get('active_survivors_mean', float('nan'))
plot_df['survivor_x'] = pd.to_numeric(plot_df['survivor_x'], errors='coerce')
has_survivor_x = plot_df['survivor_x'].notna().all() and plot_df['survivor_x'].nunique() > 1

plt.rcParams.update({
    'figure.facecolor': PAPER_BG,
    'axes.facecolor': PAPER_BG,
    'axes.edgecolor': '#9d9279',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titleweight': 'bold',
    'axes.titlesize': 24,
    'axes.labelsize': 18,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'text.color': TEXT_DARK,
    'axes.labelcolor': TEXT_DARK,
    'xtick.color': TEXT_DARK,
    'ytick.color': TEXT_DARK,
    'legend.frameon': True,
    'font.family': 'serif',
})


def _available_specs(specs, df):
    keep = []
    for prefix, label, color, marker in specs:
        col = f'{prefix}_mean'
        if col in df.columns and pd.to_numeric(df[col], errors='coerce').notna().any():
            keep.append((prefix, label, color, marker))
    return keep


def _add_bar_labels(ax, bars, std, *, offset_rank=0):
    for bar, err in zip(bars, std):
        height = bar.get_height()
        if not np.isfinite(height):
            continue
        err = 0.0 if not np.isfinite(err) else float(err)
        label_y = min(height + err + 0.028 + 0.028 * (offset_rank % 2), 1.20)
        ax.text(
            bar.get_x() + bar.get_width() / 2.0,
            label_y,
            f'{height:.2f}\n+/- {err:.2f}',
            ha='center',
            va='bottom',
            fontsize=10.5,
            fontweight='bold',
            linespacing=0.9,
            color=TEXT_DARK,
        )


def _style_axis(ax):
    ax.grid(axis='y', color=GRID_COLOR, alpha=0.55, linewidth=0.9)
    ax.grid(axis='x', color=GRID_COLOR, alpha=0.30, linewidth=0.9)
    ax.spines['left'].set_color('#9d9279')
    ax.spines['bottom'].set_color('#9d9279')


def _plot_confirm_auc_curve(ax, df):
    colors = [CONFIRM_COLOR, '#1f5a35', SCOUT_COLOR, '#5f812f', '#e09b4f']
    markers = ['o', 's', '^', 'v', 'D']
    configured = [_strategy_tag(strategy) for strategy in STRATEGIES]
    observed = [_strategy_tag(strategy) for strategy in df['strategy'].dropna().unique()]
    strategy_order = configured + [strategy for strategy in observed if strategy not in configured]
    plotted = []
    all_x = []
    all_lower = []
    all_upper = []

    for strategy_index, strategy in enumerate(strategy_order):
        group = df.loc[df['strategy'].astype(str).map(_strategy_tag).eq(strategy)].copy()
        group = group.drop_duplicates('survivor_x', keep='last').sort_values('survivor_x')
        x = pd.to_numeric(group['survivor_x'], errors='coerce').to_numpy(dtype=float)
        mean = pd.to_numeric(group['confirm_auc_mean'], errors='coerce').to_numpy(dtype=float)
        std = pd.to_numeric(group['confirm_auc_std'], errors='coerce').to_numpy(dtype=float)
        valid = np.isfinite(x) & np.isfinite(mean)
        x = x[valid]
        mean = mean[valid]
        std = np.where(np.isfinite(std[valid]), std[valid], 0.0)
        if not len(x):
            continue

        lower = np.clip(mean - std, 0.0, 1.0)
        upper = np.clip(mean + std, 0.0, 1.0)
        color = colors[strategy_index % len(colors)]
        label = STRATEGY_LABELS.get(strategy, strategy.replace('_', ' ').title())
        ax.fill_between(x, lower, upper, color=color, alpha=0.12, linewidth=0)
        ax.plot(
            x,
            mean,
            color=color,
            marker=markers[strategy_index % len(markers)],
            markersize=8,
            linewidth=3.0,
            label=label,
        )
        plotted.append(label)
        all_x.append(x)
        all_lower.append(lower)
        all_upper.append(upper)

    if not plotted:
        ax.set_axis_off()
        ax.set_title('Confirm AUC vs Survivor Count not available', color=TEXT_GREEN, pad=12)
        return

    x = np.concatenate(all_x)
    lower = np.concatenate(all_lower)
    upper = np.concatenate(all_upper)
    x_margin = max((float(np.nanmax(x)) - float(np.nanmin(x))) * 0.06, 0.5)
    y_min = max(float(np.nanmin(lower)) - 0.08, 0.0)
    y_max = min(max(float(np.nanmax(upper)) + 0.12, y_min + 0.2), 1.08)
    ax.set_xlim(float(np.nanmin(x)) - x_margin, float(np.nanmax(x)) + x_margin)
    ax.set_ylim(y_min, y_max)
    ax.set_xticks(sorted(set(x.tolist())))
    ax.set_title('Confirm AUC vs Survivors', color=TEXT_GREEN, pad=12)
    ax.set_xlabel('Survivors')
    ax.set_ylabel('Confirm AUC')
    _style_axis(ax)
    if len(plotted) > 1:
        legend = ax.legend(
            loc='center left',
            bbox_to_anchor=(1.02, 0.5),
            borderaxespad=0.0,
            fontsize=13,
        )
        legend.get_frame().set_facecolor(PAPER_BG)
        legend.get_frame().set_edgecolor('#c8bea4')
        legend.get_frame().set_alpha(0.90)


def _plot_auc_group(ax, df, specs):
    specs = _available_specs(specs, df)
    if not specs:
        ax.set_axis_off()
        ax.set_title('Joint Diagnostic Time-Integrated Performance (AUC) not available', color=TEXT_GREEN, pad=12)
        return
    positions = np.arange(len(df))
    width = min(0.155, 0.68 / max(len(specs), 1))
    for offset, (prefix, label, color, _marker) in enumerate(specs):
        mean = pd.to_numeric(df[f'{prefix}_mean'], errors='coerce').to_numpy(dtype=float)
        std = pd.to_numeric(df[f'{prefix}_std'], errors='coerce').to_numpy(dtype=float)
        bar_x = positions + (offset - (len(specs) - 1) / 2) * width
        bars = ax.bar(
            bar_x,
            mean,
            width=width,
            yerr=std,
            capsize=3,
            color=color,
            alpha=0.98,
            label=label,
            edgecolor=PAPER_BG,
            linewidth=0.8,
            error_kw={'elinewidth': 1.0, 'alpha': 0.65, 'ecolor': TEXT_DARK},
        )
        _add_bar_labels(ax, bars, std, offset_rank=offset)
    labels = df['label'].str.replace(r' \| .*$', '', regex=True).tolist()
    ax.set_xticks(positions)
    ax.set_xticklabels(labels, rotation=0, ha='center', fontweight='bold', fontsize=16)
    ax.set_title('Joint Diagnostic Time-Integrated Performance (AUC)', color=TEXT_GREEN, pad=12)
    ax.set_ylabel('Score')
    ax.set_ylim(0.0, 1.24)
    _style_axis(ax)
    legend = ax.legend(loc='lower right', title=None, fontsize=14)
    legend.get_frame().set_facecolor(PAPER_BG)
    legend.get_frame().set_edgecolor('#c8bea4')
    legend.get_frame().set_alpha(0.90)


if has_survivor_x:
    fig, ax = plt.subplots(figsize=(9.8, 5.8), facecolor=PAPER_BG)
    _plot_confirm_auc_curve(ax, plot_df)
else:
    plot_df = plot_df.reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(max(10, len(plot_df) * 1.4), 5.8), facecolor=PAPER_BG)
    _plot_auc_group(ax, plot_df, AUC_METRIC_SPECS)

fig.tight_layout(rect=(0.0, 0.0, 0.82, 1.0) if has_survivor_x else None)
plt.show()
